In [ ]:
import tidy3d as td
import tidy3d.web as web
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# MMI 1x2 SOI 220

Geometria con una entrada centrada en `y=0` y dos salidas simetricas. El `output_gap` define el espacio libre entre los bordes de los tapers de salida donde se conectan con la region ancha del MMI.

El visor de Tidy3D puede inspeccionar el objeto `sim`. El flujo posterior permite estimar el coste, ejecutar la simulacion y resumir el reparto de potencia, Excess Loss y fase. No se ejecuta ningun trabajo en la nube hasta ejecutar la celda `job.run()`.


In [ ]:
# Parametros geometricos y opticos (unidades de longitud: um).
wl_c = 1.55
mmi_length = 19.31
mmi_width = 4.8
wg_width = 0.5
wg_thick = 0.22
wg_length = 1.0
wg_mmi_dist = 6.0
mmi_input_length = 4.0
input_taper_width = 1.3
output_taper_width = 1.3
output_gap = 1.1

size_y_margin = 1.5
min_steps_per_wvl = 15
run_time_spec = {"quality_factor": 1, "source_factor": 3}
monitor_wavelengths_um = np.linspace(1.50, 1.60, 51)
monitor_freqs = td.C_0 / monitor_wavelengths_um

# Center positions of the two output waveguides, derived from their edge gap.
output_port_y = output_gap / 2 + output_taper_width / 2

SiO2 = td.Medium(name="SiO2", permittivity=2.085136)
Si = td.Medium(name="Si", permittivity=12.1104)


## Construir la geometria y la simulacion

La entrada tiene un taper centrado que pasa de `wg_width` a `input_taper_width`. A la derecha, dos tapers simetricos enlazan la region MMI con las salidas; entre sus bordes en la cara del MMI queda `output_gap`.


In [ ]:
def build_sim() -> td.Simulation:
    size_x = mmi_length + 2 * mmi_input_length + 2 * wg_mmi_dist + 2 * wg_length
    size_y = size_y_margin + mmi_width + 2 * wl_c
    size_z = wg_thick + 2 * wl_c

    port_x = mmi_length / 2 + mmi_input_length
    access_length = wg_mmi_dist + 2 * wg_length
    access_center_x = port_x + access_length / 2

    source = td.ModeSource(
        name="source_input",
        center=[-size_x / 2 + wg_length - 0.225, 0, 0],
        size=[0, 4 * wg_width, 5 * wg_thick],
        source_time=td.GaussianPulse(
            freq0=td.C_0 / wl_c,
            fwidth=12491352416666.656,
        ),
        num_freqs=5,
        direction="+",
        mode_spec=td.ModeSpec(
            num_modes=4,
            target_neff=3,
            sort_spec={
                "filter_reference": 0,
                "filter_order": "over",
                "sort_key": "n_eff",
                "track_freq": "central",
                "keep_modes": "all",
            },
            group_index_step=True,
        ),
    )

    # The monitor windows avoid including the neighboring output core.
    monitor_y_size = min(4 * wg_width, 2 * output_port_y - wg_width)
    output_x = size_x / 2 - wg_length + 0.225
    output_mode_spec = td.ModeSpec(
        num_modes=1,
        target_neff=3.48,
        sort_spec={
            "filter_reference": 0,
            "filter_order": "over",
            "sort_key": "n_eff",
            "track_freq": "central",
            "keep_modes": "all",
        },
        group_index_step=True,
    )
    monitor_upper = td.ModeMonitor(
        name="monitor_upper",
        center=[output_x, output_port_y, 0],
        size=[0, monitor_y_size, 5 * wg_thick],
        freqs=monitor_freqs,
        mode_spec=output_mode_spec,
    )
    monitor_lower = td.ModeMonitor(
        name="monitor_lower",
        center=[output_x, -output_port_y, 0],
        size=[0, monitor_y_size, 5 * wg_thick],
        freqs=monitor_freqs,
        mode_spec=output_mode_spec,
    )

    structures = [
        td.Structure(
            geometry=td.Box(
                center=[0, 0, 0],
                size=[mmi_length, mmi_width, wg_thick],
            ),
            name="mmi_slab",
            medium=Si,
        ),
        td.Structure(
            geometry=td.PolySlab(
                slab_bounds=(-wg_thick / 2, wg_thick / 2),
                vertices=[
                    (-port_x, -wg_width / 2),
                    (-mmi_length / 2, -input_taper_width / 2),
                    (-mmi_length / 2, input_taper_width / 2),
                    (-port_x, wg_width / 2),
                ],
            ),
            name="taper_input",
            medium=Si,
        ),
    ]

    # Two output tapers; their inner edges at the MMI face are separated by output_gap.
    for sign_y, label in [(1, "upper"), (-1, "lower")]:
        vertices = [
            (port_x, sign_y * (output_gap / 2 + output_taper_width / 2 + wg_width / 2)),
            (mmi_length / 2, sign_y * (output_gap / 2 + output_taper_width)),
            (mmi_length / 2, sign_y * (output_gap / 2)),
            (port_x, sign_y * (output_gap / 2 + output_taper_width / 2 - wg_width / 2)),
        ]
        structures.append(
            td.Structure(
                geometry=td.PolySlab(
                    slab_bounds=(-wg_thick / 2, wg_thick / 2),
                    vertices=vertices,
                ),
                name=f"taper_output_{label}",
                medium=Si,
            )
        )

    # Straight input extension from the centered input taper to the simulation port.
    structures.append(
        td.Structure(
            geometry=td.Box(
                center=[-access_center_x, 0, 0],
                size=[access_length, wg_width, wg_thick],
            ),
            name="access_input",
            medium=Si,
        )
    )

    # Straight output extensions from the tapers to the two simulation ports.
    for sign_y, label in [(1, "upper"), (-1, "lower")]:
        structures.append(
            td.Structure(
                geometry=td.Box(
                    center=[access_center_x, sign_y * output_port_y, 0],
                    size=[access_length, wg_width, wg_thick],
                ),
                name=f"access_output_{label}",
                medium=Si,
            )
        )

    field_xy = td.FieldMonitor(
        name="field_xy",
        size=[td.inf, td.inf, 0],
        freqs=[td.C_0 / wl_c],
    )

    return td.Simulation(
        size=[size_x, size_y, size_z],
        grid_spec=td.GridSpec.auto(
            wavelength=wl_c,
            min_steps_per_wvl=min_steps_per_wvl,
        ),
        run_time=run_time_spec,
        medium=SiO2,
        sources=[source],
        monitors=[monitor_upper, monitor_lower, field_xy],
        structures=structures,
    )

sim = build_sim()


## Estimar el coste

La estimacion devuelve el coste maximo en FlexCredits y no inicia el solver. Si cambias la geometria o el `run_time_spec`, vuelve a ejecutar las celdas de definicion y estimacion.

[API oficial de `web.Job` y `estimate_cost`](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.Job.html).

La simulacion 2x2 anterior aviso que el decaimiento final del campo superaba el umbral de parada. Este notebook empieza con el mismo ajuste temporal; si aparece de nuevo esa advertencia, trata los resultados como provisionales. Aumentar `run_time_spec` puede aumentar el coste.


In [ ]:
job = web.Job(
    simulation=sim,
    task_name="1x2_SOI220_5050",
)

estimated_cost = job.estimate_cost()
print(f"Coste maximo estimado: {estimated_cost:.3f} FlexCredits")


## Ejecutar la simulacion

Ejecuta esta celda para lanzar el solver y descargar los resultados.

[Metodo oficial `Job.run`](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.Job.html).


In [ ]:
sim_data = job.run(path="./data/1x2_sim_data.hdf5")


## Campo en el plano XY

Primero se visualiza el campo electrico en el plano del monitor XY. Tidy3D superpone la estructura en el grafico.

[API oficial `plot_field`](https://docs.flexcompute.com/projects/tidy3d/en/latest/faq/docs/faq/how-do-i-plot-the-simulation-structure-on-top-of-my-field-plot.html).


In [ ]:
ax = sim_data.plot_field(
    field_monitor_name="field_xy",
    field_name="E",
    val="abs",
    z=0,
)
ax.set_title("Electric field magnitude |E| in XY plane")
plt.show()


## Resumen de resultados

Las razones superior e inferior se normalizan por la potencia modal total de salida. La perdida excedente se calcula respecto a la potencia incidente de 1 W de `ModeSource`. La fase se define como `fase_inferior - fase_superior`, envuelta entre -180 y 180 grados.


In [ ]:
S_upper = sim_data["monitor_upper"].amps.sel(direction="+", mode_index=0)
S_lower = sim_data["monitor_lower"].amps.sel(direction="+", mode_index=0)
freqs = np.asarray(S_upper.f.values)
wavelength_um = td.C_0 / freqs

P_upper = np.abs(np.asarray(S_upper.values)) ** 2
P_lower = np.abs(np.asarray(S_lower.values)) ** 2
P_total = P_upper + P_lower

split_upper = np.divide(P_upper, P_total, out=np.full_like(P_total, np.nan, dtype=float), where=P_total > 0)
split_lower = np.divide(P_lower, P_total, out=np.full_like(P_total, np.nan, dtype=float), where=P_total > 0)
excess_loss_db = np.full_like(P_total, np.nan, dtype=float)
positive_total = P_total > 0
excess_loss_db[positive_total] = -10 * np.log10(P_total[positive_total])
phase_lower_minus_upper_deg = np.rad2deg(
    np.angle(np.asarray(S_lower.values) * np.conj(np.asarray(S_upper.values)))
)

spectrum_df = pd.DataFrame({
    "wavelength_um": wavelength_um,
    "P_upper_W": P_upper,
    "P_lower_W": P_lower,
    "split_upper": split_upper,
    "split_lower": split_lower,
    "Excess_Loss_dB": excess_loss_db,
    "phase_lower_minus_upper_deg": phase_lower_minus_upper_deg,
})

nominal_index = int(np.argmin(np.abs(wavelength_um - wl_c)))
summary_df = spectrum_df.iloc[[nominal_index]].reset_index(drop=True)
print(f"Nominal response near wl_c = {summary_df.loc[0, 'wavelength_um']:.4f} um")
display(summary_df)


## Graficos espectrales


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True, constrained_layout=True)

axes[0].plot(wavelength_um, split_upper, label="Upper output")
axes[0].plot(wavelength_um, split_lower, label="Lower output")
axes[0].set_ylabel("Output power ratio")
axes[0].set_title("1x2 power split")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(wavelength_um, excess_loss_db, color="tab:red")
axes[1].set_ylabel("Excess Loss (dB)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(wavelength_um, phase_lower_minus_upper_deg, color="tab:purple")
axes[2].set_xlabel("Wavelength (um)")
axes[2].set_ylabel("Phase (deg)")
axes[2].set_title("Lower minus upper output phase")
axes[2].grid(True, alpha=0.3)

plt.show()
